In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../tables/part_1")

consumer_path = DATA_DIR / "tbl_consumer.csv"
consumer_details_path = DATA_DIR / "consumer_user_details.parquet"
merchant_path = DATA_DIR / "tbl_merchants.parquet"

In [ ]:
consumer = pd.read_csv(
    consumer_path,
    sep="|"
)

consumer_details = pd.read_parquet(
    consumer_details_path
)

merchants = pd.read_parquet(
    merchant_path
)


In [ ]:
# compact schema summary 
#data validation section 

def schema_summary(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null": df.isna().sum(),
        "unique": df.nunique()
    })

In [ ]:
'''display(schema_summary(consumer))
display(schema_summary(consumer_details))
display(schema_summary(merchants))'''

In [ ]:
#missing value summary
#data validation section 
'''def missing_summary(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(4)
    })

display(missing_summary(consumer))
display(missing_summary(consumer_details))
display(missing_summary(merchants))

def dataset_missing_summary(df):
    return pd.Series({
        "rows": len(df),
        "columns": df.shape[1],
        "total_cells": df.size,
        "missing_cells": df.isna().sum().sum(),
        "missing_percent": round(df.isna().sum().sum() / df.size * 100, 4)
    })

summary = pd.DataFrame({
    "consumer": dataset_missing_summary(consumer),
    "consumer_details": dataset_missing_summary(consumer_details),
    "merchants": dataset_missing_summary(merchants)
}).T

display(summary)

consumer_ids = set(consumer["consumer_id"])
detail_consumer_ids = set(consumer_details["consumer_id"])

print("Consumer IDs in consumer but not details:",
      len(consumer_ids - detail_consumer_ids))

print("Consumer IDs in details but not consumer:",
      len(detail_consumer_ids - consumer_ids))
relationship_check = consumer.merge(
    consumer_details,
    on="consumer_id",
    how="outer",
    indicator=True
)

relationship_check["_merge"].value_counts()
consumer.merge(
    consumer_details,
    on="consumer_id",
    how="left",
    validate="one_to_one"
)
'''


In [ ]:
def validate_consumer_keys(consumer):
    assert consumer["consumer_id"].notna().all(), \
        "consumer_id contains missing values"

    assert consumer["consumer_id"].is_unique, \
        "consumer_id is not unique"

    print("Consumer key validation passed.")


def validate_consumer_details_keys(consumer_details):
    assert consumer_details["consumer_id"].notna().all(), \
        "consumer_details.consumer_id contains missing values"

    assert consumer_details["user_id"].notna().all(), \
        "user_id contains missing values"

    assert consumer_details["consumer_id"].is_unique, \
        "consumer_details.consumer_id is not unique"

    assert consumer_details["user_id"].is_unique, \
        "user_id is not unique"

    print("Consumer details key validation passed.")


def validate_consumer_relationship(consumer, consumer_details):
    # checks that the join is one-to-one
    consumer.merge(
        consumer_details,
        on="consumer_id",
        how="left",
        validate="one_to_one"
    )

    # checks that both tables contain exactly the same consumer IDs
    consumer_ids = set(consumer["consumer_id"])
    detail_ids = set(consumer_details["consumer_id"])

    assert consumer_ids == detail_ids, \
        "Consumer ID sets do not match between tables"

    print("Consumer relationship validation passed.")

In [ ]:
validate_consumer_keys(consumer)
validate_consumer_details_keys(consumer_details)
validate_consumer_relationship(consumer, consumer_details)

In [ ]:
print("FULL ROW DUPLICATES")
print("Consumer:", consumer.duplicated().sum())
print("Consumer details:", consumer_details.duplicated().sum())
print("Merchants:", merchants.duplicated().sum())

In [27]:
print("\nKEY DUPLICATES")

print(
    "Consumer consumer_id:",
    consumer["consumer_id"].duplicated().sum()
)

print(
    "Consumer details consumer_id:",
    consumer_details["consumer_id"].duplicated().sum()
)

print(
    "Consumer details user_id:",
    consumer_details["user_id"].duplicated().sum()
)


KEY DUPLICATES
Consumer consumer_id: 0
Consumer details consumer_id: 0
Consumer details user_id: 0


In [28]:
print(
    "Merchant ABN duplicates:",
    merchants.index.duplicated().sum()
)

print(
    "Merchant ABNs unique:",
    merchants.index.is_unique
)

print(
    "Merchant ABN missing:",
    merchants.index.isna().sum()
)

Merchant ABN duplicates: 0
Merchant ABNs unique: True
Merchant ABN missing: 0


In [29]:
print("STATE VALUES")
print(consumer["state"].value_counts(dropna=False).sort_index())

STATE VALUES
state
ACT      4664
NSW    144188
NT       7764
QLD     72861
SA      54973
TAS     18878
VIC    117525
WA      79146
Name: count, dtype: int64


In [30]:
valid_states = {
    "ACT", "NSW", "NT", "QLD",
    "SA", "TAS", "VIC", "WA"
}

invalid_states = consumer.loc[
    ~consumer["state"].isin(valid_states),
    "state"
]

print("Invalid state rows:", len(invalid_states))
print("Invalid state values:", invalid_states.unique())

Invalid state rows: 0
Invalid state values: []


In [31]:
print("GENDER VALUES")
print(consumer["gender"].value_counts(dropna=False))

GENDER VALUES
gender
Male           224979
Female         224946
Undisclosed     50074
Name: count, dtype: int64


In [15]:
postcode_as_string = consumer["postcode"].astype(str)

print("POSTCODE LENGTHS")
print(postcode_as_string.str.len().value_counts().sort_index())

POSTCODE LENGTHS
postcode
3      7909
4    492090
Name: count, dtype: int64


In [16]:
invalid_length_postcodes = consumer[
    ~postcode_as_string.str.len().isin([3, 4])
]

print(
    "Postcodes not 3 or 4 digits:",
    len(invalid_length_postcodes)
)

Postcodes not 3 or 4 digits: 0


In [17]:
three_digit_postcodes = consumer[
    postcode_as_string.str.len() == 3
]

print(
    "Number of 3-digit postcodes:",
    len(three_digit_postcodes)
)

display(
    three_digit_postcodes[
        ["state", "postcode"]
    ].head(20)
)

Number of 3-digit postcodes: 7909


,state,postcode
2,NT,862
36,NT,885
125,NT,851
183,NT,907
247,NT,822
268,NT,871
486,NT,834
535,NT,906
562,NT,813
663,NT,837


In [18]:
print(
    three_digit_postcodes["state"]
    .value_counts()
)

state
NT     7764
ACT     145
Name: count, dtype: int64


In [19]:
blank_names = consumer[
    consumer["name"].astype(str).str.strip().eq("")
]

blank_addresses = consumer[
    consumer["address"].astype(str).str.strip().eq("")
]

print("Blank names:", len(blank_names))
print("Blank addresses:", len(blank_addresses))

Blank names: 0
Blank addresses: 0


In [20]:
name_whitespace = (
    consumer["name"] != consumer["name"].str.strip()
).sum()

address_whitespace = (
    consumer["address"] != consumer["address"].str.strip()
).sum()

print("Names with surrounding whitespace:", name_whitespace)
print("Addresses with surrounding whitespace:", address_whitespace)

Names with surrounding whitespace: 0
Addresses with surrounding whitespace: 0


In [21]:
print("Merchant index name:", merchants.index.name)
print("Merchant ABN dtype:", merchants.index.dtype)

print(
    pd.Series(merchants.index.astype(str))
    .str.len()
    .value_counts()
    .sort_index()
)

Merchant index name: merchant_abn
Merchant ABN dtype: int64
merchant_abn
11    4026
Name: count, dtype: int64


In [22]:
merchant_abns = pd.Series(
    merchants.index.astype(str),
    index=merchants.index
)

invalid_abn_length = merchant_abns[
    merchant_abns.str.len() != 11
]

print(
    "Merchant ABNs not 11 digits:",
    len(invalid_abn_length)
)

Merchant ABNs not 11 digits: 0


In [23]:
print("Unique merchant names:", merchants["name"].nunique())
print("Merchant rows:", len(merchants))

print(
    "Duplicate merchant names:",
    merchants["name"].duplicated().sum()
)

Unique merchant names: 4026
Merchant rows: 4026
Duplicate merchant names: 0


In [25]:
print("Tags dtype:", merchants["tags"].dtype)
print("Unique tag values:", merchants["tags"].nunique())

display(merchants["tags"].head(5))

Tags dtype: object
Unique tag values: 3954


merchant_abn
10023283211    ((furniture, home furnishings and equipment sh...
10142254217    ([cable, satellite, and otHer pay television a...
10165489824    ([jewelry, watch, clock, and silverware shops]...
10187291046    ([wAtch, clock, and jewelry repair shops], [b]...
10192359162    ([music shops - musical instruments, pianos, a...
Name: tags, dtype: object

In [34]:
diagnostics = {
    "consumer_duplicate_rows":
        consumer.duplicated().sum(),

    "consumer_id_duplicates":
        consumer["consumer_id"].duplicated().sum(),

    "consumer_details_duplicate_rows":
        consumer_details.duplicated().sum(),

    "consumer_details_consumer_id_duplicates":
        consumer_details["consumer_id"].duplicated().sum(),

    "consumer_details_user_id_duplicates":
        consumer_details["user_id"].duplicated().sum(),

    "merchant_duplicate_rows":
        merchants.duplicated().sum(),

    "merchant_abn_duplicates":
        merchants.index.duplicated().sum(),

    "invalid_states":
        (~consumer["state"].isin(valid_states)).sum(),

    "invalid_genders":
        (~consumer["gender"].isin(valid_genders)).sum(),

    "three_digit_postcodes":
        (consumer["postcode"].astype(str).str.len() == 3).sum(),

    "blank_names":
        consumer["name"].astype(str).str.strip().eq("").sum(),

    "blank_addresses":
        consumer["address"].astype(str).str.strip().eq("").sum(),

    "invalid_abn_length":
        (pd.Series(merchants.index.astype(str)).str.len() != 11).sum()
}

display(pd.Series(diagnostics, name="count"))

NameError: name 'valid_genders' is not defined

In [32]:
def clean_consumer(df):
    df = df.copy()

    # Postcodes are identifiers rather than numeric measurements.
    # Restore leading zeroes lost during CSV parsing.
    df["postcode"] = (
        df["postcode"]
        .astype(str)
        .str.zfill(4)
    )

    return df

In [33]:
consumer_clean = clean_consumer(consumer)

In [35]:
consumer_fraud_path = DATA_DIR / "consumer_fraud_probability.csv"
merchant_fraud_path = DATA_DIR / "merchant_fraud_probability.csv"

In [36]:
consumer_fraud = pd.read_csv(consumer_fraud_path)
merchant_fraud = pd.read_csv(merchant_fraud_path)

In [37]:
print("Consumer fraud shape:", consumer_fraud.shape)
print("Merchant fraud shape:", merchant_fraud.shape)

display(consumer_fraud.head())
display(merchant_fraud.head())
print("CONSUMER FRAUD COLUMNS")
print(consumer_fraud.columns.tolist())

print("\nMERCHANT FRAUD COLUMNS")
print(merchant_fraud.columns.tolist())

print("\nCONSUMER FRAUD DTYPES")
print(consumer_fraud.dtypes)

print("\nMERCHANT FRAUD DTYPES")
print(merchant_fraud.dtypes)

Consumer fraud shape: (34864, 3)
Merchant fraud shape: (114, 3)


,user_id,order_datetime,fraud_probability
0,6228,2021-12-19,97.629808
1,21419,2021-12-10,99.247380
2,5606,2021-10-17,84.058250
3,3101,2021-04-17,91.421921
4,22239,2021-10-19,94.703425


,merchant_abn,order_datetime,fraud_probability
0,19492220327,2021-11-28,44.403659
1,31334588839,2021-10-02,42.755301
2,19492220327,2021-12-22,38.867790
3,82999039227,2021-12-19,94.134700
4,90918180829,2021-09-02,43.325517


CONSUMER FRAUD COLUMNS
['user_id', 'order_datetime', 'fraud_probability']

MERCHANT FRAUD COLUMNS
['merchant_abn', 'order_datetime', 'fraud_probability']

CONSUMER FRAUD DTYPES
user_id                int64
order_datetime        object
fraud_probability    float64
dtype: object

MERCHANT FRAUD DTYPES
merchant_abn           int64
order_datetime        object
fraud_probability    float64
dtype: object


In [38]:
display(schema_summary(consumer_fraud))
display(schema_summary(merchant_fraud))

,dtype,non_null,null,unique
user_id,int64,34864,0,20128
order_datetime,object,34864,0,365
fraud_probability,float64,34864,0,34765


,dtype,non_null,null,unique
merchant_abn,int64,114,0,61
order_datetime,object,114,0,64
fraud_probability,float64,114,0,113


In [39]:
print("FULL ROW DUPLICATES")

print(
    "Consumer fraud:",
    consumer_fraud.duplicated().sum()
)

print(
    "Merchant fraud:",
    merchant_fraud.duplicated().sum()
)

FULL ROW DUPLICATES
Consumer fraud: 99
Merchant fraud: 0


In [40]:
print(
    "Consumer fraud ID columns:",
    [col for col in consumer_fraud.columns if "id" in col.lower()]
)

print(
    "Merchant fraud ID columns:",
    [col for col in merchant_fraud.columns if "id" in col.lower()]
)

Consumer fraud ID columns: ['user_id']
Merchant fraud ID columns: []


In [41]:
for col in consumer_fraud.columns:
    if "id" in col.lower():
        print(
            col,
            "rows:", len(consumer_fraud),
            "unique:", consumer_fraud[col].nunique(),
            "missing:", consumer_fraud[col].isna().sum()
        )

user_id rows: 34864 unique: 20128 missing: 0


In [42]:
for col in merchant_fraud.columns:
    if "id" in col.lower():
        print(
            col,
            "rows:", len(merchant_fraud),
            "unique:", merchant_fraud[col].nunique(),
            "missing:", merchant_fraud[col].isna().sum()
        )

In [43]:
print("CONSUMER FRAUD PROBABILITY")
print(consumer_fraud["fraud_probability"].describe())

print("\nMERCHANT FRAUD PROBABILITY")
print(merchant_fraud["fraud_probability"].describe())

CONSUMER FRAUD PROBABILITY
count    34864.000000
mean        15.120091
std          9.946085
min          8.287144
25%          9.634437
50%         11.735624
75%         16.216158
max         99.247380
Name: fraud_probability, dtype: float64

MERCHANT FRAUD PROBABILITY
count    114.000000
mean      40.419335
std       17.187745
min       18.210891
25%       28.992765
50%       32.692032
75%       48.395260
max       94.134700
Name: fraud_probability, dtype: float64


In [44]:
print(
    "Consumer fraud probability range:",
    consumer_fraud["fraud_probability"].min(),
    "to",
    consumer_fraud["fraud_probability"].max()
)

print(
    "Merchant fraud probability range:",
    merchant_fraud["fraud_probability"].min(),
    "to",
    merchant_fraud["fraud_probability"].max()
)

Consumer fraud probability range: 8.287143531552802 to 99.24738020302328
Merchant fraud probability range: 18.21089142894488 to 94.1347004808891


In [45]:
consumer_invalid_prob = consumer_fraud[
    ~consumer_fraud["fraud_probability"].between(0, 100)
]

merchant_invalid_prob = merchant_fraud[
    ~merchant_fraud["fraud_probability"].between(0, 100)
]

print("Invalid consumer probabilities:", len(consumer_invalid_prob))
print("Invalid merchant probabilities:", len(merchant_invalid_prob))

Invalid consumer probabilities: 0
Invalid merchant probabilities: 0


In [46]:
print("Consumer date dtype:",
      consumer_fraud["order_datetime"].dtype)

print("Merchant date dtype:",
      merchant_fraud["order_datetime"].dtype)

print("\nConsumer date examples:")
print(consumer_fraud["order_datetime"].head())

print("\nMerchant date examples:")
print(merchant_fraud["order_datetime"].head())

Consumer date dtype: object
Merchant date dtype: object

Consumer date examples:
0    2021-12-19
1    2021-12-10
2    2021-10-17
3    2021-04-17
4    2021-10-19
Name: order_datetime, dtype: object

Merchant date examples:
0    2021-11-28
1    2021-10-02
2    2021-12-22
3    2021-12-19
4    2021-09-02
Name: order_datetime, dtype: object


In [47]:
consumer_dates = pd.to_datetime(
    consumer_fraud["order_datetime"],
    errors="coerce",
    dayfirst=True
)

merchant_dates = pd.to_datetime(
    merchant_fraud["order_datetime"],
    errors="coerce",
    dayfirst=True
)

print(
    "Invalid consumer dates:",
    consumer_dates.isna().sum()
)

print(
    "Invalid merchant dates:",
    merchant_dates.isna().sum()
)

print(
    "Consumer date range:",
    consumer_dates.min(),
    "to",
    consumer_dates.max()
)

print(
    "Merchant date range:",
    merchant_dates.min(),
    "to",
    merchant_dates.max()
)

Invalid consumer dates: 0
Invalid merchant dates: 0
Consumer date range: 2021-02-28 00:00:00 to 2022-02-27 00:00:00
Merchant date range: 2021-03-25 00:00:00 to 2022-02-27 00:00:00


/var/folders/sn/d14740k16x59yrww8r0mdpch0000gn/T/ipykernel_27434/1883715745.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  consumer_dates = pd.to_datetime(
/var/folders/sn/d14740k16x59yrww8r0mdpch0000gn/T/ipykernel_27434/1883715745.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  merchant_dates = pd.to_datetime(


In [48]:
fraud_user_ids = set(consumer_fraud["user_id"])
known_user_ids = set(consumer_details["user_id"])

print(
    "Fraud user IDs not found in consumer details:",
    len(fraud_user_ids - known_user_ids)
)

Fraud user IDs not found in consumer details: 0


In [49]:
print(
    "Known users represented in fraud data:",
    len(fraud_user_ids & known_user_ids)
)

print(
    "Unique fraud users:",
    len(fraud_user_ids)
)

Known users represented in fraud data: 20128
Unique fraud users: 20128


In [50]:
fraud_merchant_ids = set(
    merchant_fraud["merchant_abn"]
)

known_merchant_ids = set(
    merchants.index
)

print(
    "Fraud merchant IDs not found in merchant table:",
    len(fraud_merchant_ids - known_merchant_ids)
)

print(
    "Known merchants represented in fraud data:",
    len(fraud_merchant_ids & known_merchant_ids)
)

print(
    "Unique fraud merchants:",
    len(fraud_merchant_ids)
)

Fraud merchant IDs not found in merchant table: 13
Known merchants represented in fraud data: 48
Unique fraud merchants: 61


In [51]:
fraud_merchant_ids = set(
    merchant_fraud["merchant_abn"]
)

known_merchant_ids = set(
    merchants.index
)

print(
    "Fraud merchant IDs not found in merchant table:",
    len(fraud_merchant_ids - known_merchant_ids)
)

print(
    "Known merchants represented in fraud data:",
    len(fraud_merchant_ids & known_merchant_ids)
)

print(
    "Unique fraud merchants:",
    len(fraud_merchant_ids)
)

Fraud merchant IDs not found in merchant table: 13
Known merchants represented in fraud data: 48
Unique fraud merchants: 61


In [ ]:

# investigating the data problems ?!?!
# Check the 99 duplicate rows in consumer fraud data
duplicate_consumer_fraud = consumer_fraud[
    consumer_fraud.duplicated(keep=False)
].sort_values(
    ["user_id", "order_datetime", "fraud_probability"]
)

display(duplicate_consumer_fraud)

,user_id,order_datetime,fraud_probability
9,230,2021-08-28,86.283288
108,230,2021-08-28,86.283288
38,251,2021-12-31,82.970975
137,251,2021-12-31,82.970975
81,587,2021-12-14,65.740496
...,...,...,...
127,23306,2021-10-23,85.582704
82,23700,2021-08-26,75.165555
181,23700,2021-08-26,75.165555
97,24058,2021-11-23,63.834305


In [53]:
print("Rows before:", len(consumer_fraud))
print("Unique rows:", len(consumer_fraud.drop_duplicates()))
print("Rows removed:", len(consumer_fraud) - len(consumer_fraud.drop_duplicates()))

Rows before: 34864
Unique rows: 34765
Rows removed: 99


In [55]:
consumer_fraud = consumer_fraud.drop_duplicates()
unmatched_merchant_abns = (
    set(merchant_fraud["merchant_abn"])
    - set(merchants.index)
)

print("Unmatched ABNs:", len(unmatched_merchant_abns))
print(unmatched_merchant_abns)

Unmatched ABNs: 13
{19010030815, 83220249221, 29674997261, 81146325646, 75892370170, 99989036621, 73052515151, 14827550074, 82999039227, 94311056026, 23686790459, 57564805948, 59258669983}


In [56]:
unmatched_merchant_fraud = merchant_fraud[
    merchant_fraud["merchant_abn"].isin(unmatched_merchant_abns)
]

display(
    unmatched_merchant_fraud.sort_values(
        ["merchant_abn", "order_datetime"]
    )
)

,merchant_abn,order_datetime,fraud_probability
7,14827550074,2021-11-26,46.457756
11,14827550074,2021-12-05,43.855195
26,14827550074,2021-12-11,39.406448
35,14827550074,2021-12-12,38.282869
43,19010030815,2021-09-28,56.806143
92,19010030815,2021-11-11,47.652943
81,19010030815,2021-12-24,48.642699
6,23686790459,2021-12-10,79.454344
56,29674997261,2021-12-26,44.437878
52,57564805948,2021-11-23,31.268145


In [57]:
print("Unmatched merchant ABNs:",
      len(unmatched_merchant_abns))

print("Fraud rows belonging to unmatched merchants:",
      len(unmatched_merchant_fraud))

print(
    "Percentage of merchant fraud rows unmatched:",
    round(
        len(unmatched_merchant_fraud)
        / len(merchant_fraud) * 100,
        2
    ),
    "%"
)

Unmatched merchant ABNs: 13
Fraud rows belonging to unmatched merchants: 19
Percentage of merchant fraud rows unmatched: 16.67 %


In [58]:
# merchant investigation 
merchant_abns = set(merchants_clean.index)
fraud_merchant_abns = set(merchant_fraud_clean["merchant_abn"])

print(
    "Fraud merchants not in merchant data:",
    len(fraud_merchant_abns - merchant_abns)
)

print(
    "Unique fraud merchants:",
    len(fraud_merchant_abns)
)

print(
    "Merchant fraud rows:",
    len(merchant_fraud_clean)
)

NameError: name 'merchants_clean' is not defined